In [1]:
import os 
import h5py 
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
from gecatsim.pyfiles.GetMu import GetMu

ModuleNotFoundError: No module named 'pandas'

In [ ]:
mat_folder = r"C:\Users\Nelly Kleppe\PycharmProjects\exjobb\calibration\matfiles" # folder with .mat files 
save_folder = r"C:\Users\Nelly Kleppe\PycharmProjects\exjobb\calibration" 
save_file = "nelly_test.xlsx" # output Excel file 
kev_values = list(range(40, 60, 10)) 
files = ['head1.mat', 'head2.mat', 'head3.mat', 'head4.mat', 'head5.mat', 'body1.mat', 'body2.mat', 'body3.mat', 'body4.mat', 'body5.mat'] 
r=8

In [ ]:
def mean_in_circle(img, cx_mm, cy_mm, r_mm, fov): 
    cx, cy, r = mm_centered_to_pixel(cx_mm, cy_mm, r_mm, img.shape, fov) 
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]] 
    mask = (xx - cx)**2 + (yy - cy)**2 <= r**2 
    return np.mean(img[mask]) 

def mm_centered_to_pixel(cx_mm, cy_mm, r_mm, img_shape, fov): 
    ny, nx = img_shape 
    px_size = fov/1024 
    # image center in pixel coordinates 
    x0 = (nx - 1) / 2 
    y0 = (ny - 1) / 2 
    # convert mm to pixel 
    x_px = x0 + cx_mm / px_size 
    y_px = y0 - cy_mm / px_size 
    r_px = r_mm / px_size 
    return int(round(x_px)), int(round(y_px)), r_px

In [ ]:
# Defining geometries, materials and ROI centers for body and head 
material_names = [None, 'Schneider_gammex_HE_CT_Solid_Water', 'Schneider_gammex_Liquid_Water', 'Schneider_gammex_HE_Brain', 'Schneider_gammex_HE_Breast_5050', 'Schneider_gammex_HE_General_Adipose', 'Schneider_gammex_HE_Liver', 'Schneider_gammex_LN300_Lung', 'Schneider_gammex_LN450_Lung', 'Schneider_gammex_HE_Inner_Bone', 'Schneider_gammex_CB2_30pct_CaCO3_Bone', 'Schneider_gammex_CB2_50pct_CaCO3_Bone', 'Schneider_gammex_HE_Cortical_Bone' ] 
size_body = [200,150] # half-axes 
cx_body = [0, 0, 53.03, 73.333300, 53.03, 0, -53.03, -73.333300, -53.03, 125.85, 150, 125.85, -125.85, 0, -125.85] 
cy_body = [37.5, 73.333300, 53.03, 0, -53.03, -73.333300, -53.03, 0, 53.03, 71, 0, -71, -71, -150, 71] 
insert_material_body = [1, 1, 6, 1, 1, 7, 1, 5, 1, 1, 1, 2, 3, 4, 8] 
body_pos = pd.DataFrame({"cx":cx_body, "cy":cy_body, "insert_material":insert_material_body}) # separate copies 
body1 = body_pos.copy() 
body2 = body_pos.copy() 
body3 = body_pos.copy() 
body4 = body_pos.copy() 
body5 = body_pos.copy() 

# add unique middle inserts 
body1.loc[len(body1)] = [0, 0, 1] 
body2.loc[len(body2)] = [0, 0, 9] 
body3.loc[len(body3)] = [0, 0, 10] 
body4.loc[len(body4)] = [0, 0, 11] 
body5.loc[len(body5)] = [0, 0, 12] 
pos_map = { "body1.mat": body1, "body2.mat": body2, "body3.mat": body3, "body4.mat": body4, "body5.mat": body5, } 


size_head = [100, 100] # half-axes 
cx_head = [0, 0, 53.03, 75, 53.03, 0, -53.03, -75, -53.03] 
cy_head = [37.5, 75, 53.03, 0, -53.03, -75, -53.03, 0, 53.03] 
insert_material_head = [1, 4, 6, 1, 2, 7, 3, 5, 8] 
head_pos = pd.DataFrame({"cx":cx_head, "cy":cy_head, "insert_material":insert_material_head}) # separate copies 
head1 = head_pos.copy() 
head2 = head_pos.copy() 
head3 = head_pos.copy() 
head4 = head_pos.copy() 
head5 = head_pos.copy() 

# add unique middle inserts 
head1.loc[len(head1)] = [0, 0, 1] 
head2.loc[len(head2)] = [0, 0, 9] 
head3.loc[len(head3)] = [0, 0, 10] 
head4.loc[len(head4)] = [0, 0, 11] 
head5.loc[len(head5)] = [0, 0, 12] 
pos_map.append{ "head1.mat": head1, "head2.mat": head2, "head3.mat": head3, "head4.mat": head4, "head5.mat": head5, }


In [ ]:
rows = []

mu_PE = GetMu('polyethylene', kev_values)
mu_PVC = GetMu('pvc_legacy', kev_values)
mu_water = GetMu('water', kev_values)
mu_air = GetMu('air', kev_values)

for file in files:
    file_path = os.path.join(mat_folder, file)
    with h5py.File(file_path, 'r') as f:
        pe = 10*np.array(f['Image_PE'])
        pvc = 10*np.array(f['Image_PVC'])
    pe = np.flipud(pe)
    pvc = np.flipud(pvc)
    pos = pos_map[file]
    if str(file).startswith('head'):
        fov = 350
    elif str(file).startswith('body'):
        fov = 420
    for e in range(len(kev_values)):
        mu_pe = mu_PE[e]
        mu_pvc = mu_PVC[e]
        mu_w = mu_water[e]
        mu = mu_pe*pe+mu_pvc*pvc
        mu_a = mu_air[e]
        HU = 1000*(mu-mu_w)/(mu_w-mu_a)
        for i in range(len(pos)):
            cx = (pos.iloc[i]["cx"])
            cy = (pos.iloc[i]["cy"])
            material = pos.iloc[i]["insert_material"]
            val = mean_in_circle(HU, cx, cy, r, fov)

            rows.append({
                "kev": kev_values[e],
                "insert_material": material,
                "mean_hu_roi": val
            })

df = pd.DataFrame(rows)
df.head()

In [ ]:
# average for each material and energy 
output = ( df .groupby(["kev", "insert_material"], as_index=False)["mean_hu_roi"] .mean() ) 

# map index to name and overwrite the column 
material_map = dict(enumerate(material_names)) 
output["insert_material"] = output["insert_material"].astype(int).map(material_map) # (optional) rename column for clarity 
output = output.rename(columns={"insert_material": "material_name"}) 
output.head()

In [ ]:
# export to excel 
output.to_excel(os.path.join(save_folder, save_file), index=False)

In [ ]:
# plot a VMI from matplotlib.patches import Rectangle E = 120 mu_pe = GetMu('polyethylene', E) mu_pvc = GetMu('pvc_flexible', E) mu_w = GetMu('water', E) file_path = r'C:\Users\User1\Desktop\lutpy-main\md_files\head1.mat' with h5py.File(file_path, 'r') as f: pe = 10*np.array(f['Image_PE']) pvc = 10*np.array(f['Image_PVC']) mu = mu_pe*pe+mu_pvc*pvc HU = 1000*(mu-mu_w)/mu_w mu = mu_pe * pe + mu_pvc * pvc HU = 1000 * (mu - mu_w) / mu_w plt.figure(figsize=(6, 6)) plt.imshow(HU, cmap='gray') plt.colorbar(label='HU') plt.title(f'HU at {E} keV') plt.axis('off') mid = int(1024/2) rad = 20 x = 650 y = mid # draw square ROI rect = Rectangle( (x - rad, y - rad), # bottom-left corner 2 * rad, # width 2 * rad, # height edgecolor='red', facecolor='none', linewidth=2 ) plt.gca().add_patch(rect) plt.show() mid = int(1024/2) print(np.mean(HU[0:rad, 0:rad])) print(np.mean(HU[mid-rad:mid+rad, mid-rad:mid+rad])) print(np.log(np.var(HU[0:rad, 0:rad]))) print(np.log(np.var(HU[mid-rad:mid+rad, mid-rad:mid+rad])))